In [1]:
import os
import re

import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.mask import mask


# ============================================================
# 0. PROJECT STRUCTURE
# ============================================================

BASE_DIR = os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR = os.path.join(BASE_DIR, "output")
WEIGHT_DIR = os.path.join(BASE_DIR, "weights")
SRC_DIR = os.path.join(BASE_DIR, "src")
SHP_DIR = os.path.join(BASE_DIR, "shp")

for d in [DATA_DIR, OUT_DIR, WEIGHT_DIR, SRC_DIR, SHP_DIR]:
    os.makedirs(d, exist_ok=True)

print("PROJECT STRUCTURE READY.")
print("BASE_DIR:", BASE_DIR)


# ============================================================
# 1. CLIP RASTERS BY NEIGHBORHOOD SHAPEFILE
# ============================================================

def clip_raster_by_neighborhoods(shp_path, raster_path, output_dir):
    """
    Clip a raster by each neighborhood polygon in the shapefile.
    Output filenames use only buurtcode.
    """
    os.makedirs(output_dir, exist_ok=True)

    shapes = gpd.read_file(shp_path)

    print("\n=== CLIPPING RASTER ===")
    print("Input raster:", raster_path)
    print("Output directory:", output_dir)

    for idx, row in shapes.iterrows():
        geom = [row["geometry"]]
        buurtcode = re.sub(r"\W+", "_", str(row.get("buurtcode", f"unknown_{idx}")))

        out_name = f"{buurtcode}.tif"
        out_path = os.path.join(output_dir, out_name)

        with rasterio.open(raster_path) as src:
            out_img, out_transform = mask(src, geom, crop=True)
            meta = src.meta.copy()
            meta.update({
                "driver": "GTiff",
                "height": out_img.shape[1],
                "width": out_img.shape[2],
                "transform": out_transform
            })

        with rasterio.open(out_path, "w", **meta) as dest:
            dest.write(out_img)

        print("Saved:", out_path)

    print("Raster clipping completed.")
    return output_dir


# ============================================================
# 2. GENERATE FRAGSTATS FBT FILES
# ============================================================

def generate_fbt(clip_dir, frag_dir, fbt_name):
    """
    Generate a Fragstats batch file (.fbt) from clipped TIFF files.

    Parameters:
    - clip_dir: directory containing clipped TIFF files
    - frag_dir: Fragstats working/output directory
    - fbt_name: output FBT filename
    """
    os.makedirs(frag_dir, exist_ok=True)

    fbt_path = os.path.join(frag_dir, fbt_name)
    template = ", x, 999, x, x, 1, x, IDF_GeoTIFF"

    lines = []

    for root, dirs, files in os.walk(clip_dir):
        for file in files:
            if file.lower().endswith(".tif"):
                tif_path = os.path.join(root, file)
                lines.append(tif_path + template)

    with open(fbt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("FBT file generated:", fbt_path)
    return fbt_path


# ============================================================
# 3. HELPER FUNCTIONS FOR FRAGSTATS RESULTS
# ============================================================

def clean_columns(df):
    """
    Clean column names by removing BOM, leading/trailing spaces,
    and internal spaces.
    """
    df.columns = (
        df.columns
        .astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.replace(" ", "", regex=False)
    )
    return df


def extract_buurtcode(path_str):
    """
    Extract buurtcode from the Fragstats LID or FILE field.

    Supported filename examples:
    - BU00140000_Binnenstad_Noord_Groningen.tif
    - BU00140000.tif
    """
    s = str(path_str)
    filename = os.path.basename(s)
    first = filename.split("_")[0]
    code = first.split(".")[0]
    return code


def find_id_column(df):
    """
    Automatically detect the ID column in Fragstats output.
    Priority: LID, FILE, File, FID, Layer, LAYER.
    If none are found, use the first column.
    """
    candidates = ["LID", "FILE", "File", "FID", "Layer", "LAYER"]

    for c in candidates:
        if c in df.columns:
            return c

    return df.columns[0]


def pick_latest_csv(frag_dir, suffix):
    """
    Pick the latest CSV file by modification time.

    Parameters:
    - frag_dir: Fragstats output directory
    - suffix: expected file suffix, such as '_class.csv' or '_land.csv'
    """
    files = [
        f for f in os.listdir(frag_dir)
        if f.endswith(suffix)
    ]

    if not files:
        return None

    files_full = [os.path.join(frag_dir, f) for f in files]
    latest = max(files_full, key=os.path.getmtime)
    return latest


# ============================================================
# 4. PROCESS FRAGSTATS RESULTS
# ============================================================

def process_fragstats_results(frag_dir, pivot_cols, output_name):
    """
    Process Fragstats class-level and landscape-level results.

    Parameters:
    - frag_dir: Fragstats output directory containing .class and/or .land files
    - pivot_cols: class-level metrics, such as ['PLAND', 'AI', 'IJI']
    - output_name: final merged output filename
    """
    print("\n=== PROCESSING FRAGSTATS RESULTS ===")
    print("Directory:", frag_dir)

    for file in os.listdir(frag_dir):
        if file.endswith(".class") or file.endswith(".land"):
            input_path = os.path.join(frag_dir, file)
            base = file.rsplit(".", 1)[0]

            if file.endswith(".class"):
                out_name = f"{base}_class.csv"
            else:
                out_name = f"{base}_land.csv"

            output_path = os.path.join(frag_dir, out_name)

            df_raw = pd.read_csv(input_path, sep=None, engine="python")
            df_raw.to_csv(output_path, index=False, encoding="utf-8-sig")

            print(f"Converted: {file} -> {out_name}")

    class_path = pick_latest_csv(frag_dir, "_class.csv")
    land_path = pick_latest_csv(frag_dir, "_land.csv")

    if class_path is None:
        raise FileNotFoundError(f"No *_class.csv file found in {frag_dir}")

    print("Class file:", os.path.basename(class_path))

    if land_path:
        print("Land file:", os.path.basename(land_path))
    else:
        print("No land file found. Only class-level results will be used.")

    df_class = pd.read_csv(class_path)
    df_class = clean_columns(df_class)

    df_land = None
    if land_path is not None:
        df_land = pd.read_csv(land_path)
        df_land = clean_columns(df_land)

    id_col_class = find_id_column(df_class)
    df_class["buurtcode"] = df_class[id_col_class].apply(extract_buurtcode)
    df_class = df_class.drop(columns=[id_col_class])

    if df_land is not None:
        id_col_land = find_id_column(df_land)
        df_land["buurtcode"] = df_land[id_col_land].apply(extract_buurtcode)
        df_land = df_land.drop(columns=[id_col_land])

    df_pivot = df_class.pivot_table(
        index="buurtcode",
        columns="TYPE",
        values=pivot_cols,
        aggfunc="first"
    )

    df_pivot.columns = [
        f"{metric}-{cls}"
        for metric, cls in df_pivot.columns
    ]

    df_pivot.columns = df_pivot.columns.str.replace(" ", "", regex=False)
    df_pivot.reset_index(inplace=True)

    if df_land is not None:
        df_merged = df_land.merge(df_pivot, on="buurtcode", how="left")
    else:
        df_merged = df_pivot

    output_path = os.path.join(frag_dir, output_name)
    df_merged.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("Processed Fragstats output saved:", output_path)
    return output_path


# ============================================================
# 5. CLEAN AND RENAME MERGED FORM DATA
# ============================================================

def clean_merged_form_data(input_path, output_path):
    """
    Clean and rename columns in the merged LGN + GREEN form dataset.
    """
    print("\n=== CLEANING MERGED FORM DATA ===")

    df = pd.read_csv(input_path)
    df.columns = df.columns.str.strip()

    df = df.rename(columns={
        "AI-water": "AI_water",
        "AI-built": "AI_built",
        "AI-highproportion_mostlyshort": "AI_Dense_Short",
        "AI-highproportion_mostlytall": "AI_Dense_Tall",
        "AI-lowproportion_mostlyshort": "AI_Sparse_Short",
        "AI-lowproportion_mostlytall": "AI_Sparse_Tall",

        "IJI-built": "IJI_built",
        "IJI-water": "IJI_water",
        "IJI-highproportion_mostlyshort": "IJI_Dense_Short",
        "IJI-highproportion_mostlytall": "IJI_Dense_Tall",
        "IJI-lowproportion_mostlyshort": "IJI_Sparse_Short",
        "IJI-lowproportion_mostlytall": "IJI_Sparse_Tall",

        "PLAND-water": "PLAND_water",
        "PLAND-built": "PLAND_built",
        "PLAND-agriculture": "PLAND_agriculture",
        "PLAND-highproportion_mostlyshort": "PLAND_Dense_Short",
        "PLAND-highproportion_mostlytall": "PLAND_Dense_Tall",
        "PLAND-lowproportion_mostlyshort": "PLAND_Sparse_Short",
        "PLAND-lowproportion_mostlytall": "PLAND_Sparse_Tall",
    })

    drop_cols = [
        "0", "B", "C", "D", "E", "F", "G", "A!",
        "ste_oad", "ste_mvs", "a_opp_ha",
        "PD_y", "CONTAG_y", "PRD_y", "SHDI_y", "AI_y"
    ]

    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    pattern_cols = [c for c in df.columns if "-others" in c or "-cls_5" in c]
    df.drop(columns=pattern_cols, inplace=True)

    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("Cleaned merged form data saved:", output_path)
    print("Rows:", len(df))

    return output_path


# ============================================================
# 6. MAIN WORKFLOW
# ============================================================

def main():
    shp_path = os.path.join(DATA_DIR, "main_buurten selection.shp")

    lgn_path = os.path.join(DATA_DIR, "Reclass_LGN22.tif")
    green_path = os.path.join(DATA_DIR, "Green_4+1.tif")

    clip_lgn_dir = os.path.join(OUT_DIR, "clipped_rasters")
    clip_green_dir = os.path.join(OUT_DIR, "clipped_rasters2")

    frag_lgn_dir = os.path.join(OUT_DIR, "4frg_5types")
    frag_green_dir = os.path.join(OUT_DIR, "4frg_4green")

    clip_raster_by_neighborhoods(
        shp_path=shp_path,
        raster_path=lgn_path,
        output_dir=clip_lgn_dir
    )

    clip_raster_by_neighborhoods(
        shp_path=shp_path,
        raster_path=green_path,
        output_dir=clip_green_dir
    )

    generate_fbt(
        clip_dir=clip_lgn_dir,
        frag_dir=frag_lgn_dir,
        fbt_name="import_LGN.fbt"
    )

    generate_fbt(
        clip_dir=clip_green_dir,
        frag_dir=frag_green_dir,
        fbt_name="import_GREEN.fbt"
    )

    print("\nManual Fragstats step required:")
    print("1. Open Fragstats.")
    print("2. Import import_LGN.fbt and import_GREEN.fbt.")
    print("3. Run class-level metrics: PLAND, AI, IJI.")
    print("4. Run landscape-level metrics: PD, SHDI.")
    print("5. Export the Fragstats results into the corresponding output folders.")
    print("6. Then run the Fragstats result processing section below.")

    lgn_output = process_fragstats_results(
        frag_dir=frag_lgn_dir,
        pivot_cols=["PLAND", "IJI", "AI"],
        output_name="LGN_merged_class_land.csv"
    )

    green_output = process_fragstats_results(
        frag_dir=frag_green_dir,
        pivot_cols=["PLAND", "AI"],
        output_name="GREEN_merged_class.csv"
    )

    df_lgn = pd.read_csv(lgn_output)
    df_green = pd.read_csv(green_output)

    df_final = df_lgn.merge(df_green, on="buurtcode", how="left")

    final_path = os.path.join(OUT_DIR, "form_merged_green_again_all.csv")
    df_final.to_csv(final_path, index=False, encoding="utf-8-sig")

    print("Final LGN + GREEN merged file saved:", final_path)

    clean_final_path = os.path.join(OUT_DIR, "form_merged_green_cleaned.csv")
    clean_merged_form_data(final_path, clean_final_path)

    print("\nAll raster clipping, FBT generation, and Fragstats result processing steps completed.")


if __name__ == "__main__":
    main()

C:\Users\Yu Xiaohe\AppData\Local\Temp\ipykernel_14048\2303344456.py:4: DeprecationWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas still uses PyGEOS by default. However, starting with version 0.14, the default will switch to Shapely. To force to use Shapely 2.0 now, you can either uninstall PyGEOS or set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In the next release, GeoPandas will switch to using Shapely by default, even if PyGEOS is installed. If you only have PyGEOS installed to get speed-ups, this switch should be smooth. However, if you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


PROJECT STRUCTURE READY.
BASE_DIR: d:\1st\code\project

=== CLIPPING RASTER ===
Input raster: d:\1st\code\project\data\Reclass_LGN22.tif
Output directory: d:\1st\code\project\output\clipped_rasters
Saved: d:\1st\code\project\output\clipped_rasters\BU00140000.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140001.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140002.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140003.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140004.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140005.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140007.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140008.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140100.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140101.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140102.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140103.tif
Saved: d:\1st\code\pro